### Simple Gen AI APP using LangChain

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["USER_AGENT"] = "my-langchain-app"

## Langsmith Tracking Keys, 3 keys
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]="true"

In [15]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [17]:
## Data Ingestion--From the website we need to scrape the data (Data Scraping) 
# and then we need to split the data into chunks (Data Splitting) 
# and then we need to embed the data into vectors (Data Embedding) 
# and then we need to store the data into a vector database (Data Storage) 
# and then we can query the data from the vector database (Data Querying)
loader = WebBaseLoader("https://docs.langchain.com/langsmith/administration-overview")
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'description': 'Learn how LangSmith organizes organizations, workspaces, and resources, and how users, API keys, and roles work.', 'language': 'en'}, page_content='\n\n\n\n\n\nOverview - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageLangSmith setupSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationOrganizationOverviewOverviewAccountCloudBYOCSelf-hostedGovernOverviewOrganizationOverviewWorkspace setupTunnelsBetaRelaysBetaManage organizations using the APITerraform providerUsers & access controlUser managementRole-based access controlAttribut

In [18]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'description': 'Learn how LangSmith organizes organizations, workspaces, and resources, and how users, API keys, and roles work.', 'language': 'en'}, page_content='Overview - Docs by LangChain'),
 Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'description': 'Learn how LangSmith organizes organizations, workspaces, and resources, and how users, API keys, and roles work.', 'language': 'en'}, page_content='Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageLangSmith setupSearch...⌘KAsk AIGitHubTry LangSmithTry Lang

In [6]:
# dcouments-->text-->vectors-->Vector Embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
embedded_docs = embeddings.embed_documents([d.page_content for d in documents])
# embedded_docs

In [ ]:
# Vector Store DB
# documents u have to pass the documents 
# and embeddings to apply to create a vector store database
vectorstoredb=FAISS.from_documents(documents,embeddings)
# vectorstoredb

In [9]:
## Query From a vector db
query="What is organization?"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Organizations\nAn organization is a logical grouping of users within LangSmith that defines shared settings applying across all of its workspaces. These settings govern organization-wide concerns rather than the agents and resources within a workspace. Common organization-level configurations include user management, single sign-on (SSO), OAuth provider configuration, custom role creation, billing, and usage tracking.\nTypically, there is one organization per company. An organization can have multiple workspaces. For configuration details, see Set up an organization.\nWhen you log in for the first time, a personal organization will be created for you automatically. If you’d like to collaborate with others, you can create a separate organization and invite your team members to join. There are a few important differences between your personal organization and shared organizations:'

In [20]:
llm = ChatOllama(
    model="gpt-oss:20b",   # exactly as shown in `ollama list`
    temperature=0.7,
    # base_url="http://localhost:11434",  # only needed if Ollama runs elsewhere
)

In [23]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>

Question: {input}
"""
)
# DOCUMENT CHAIN: joins the prompt and the LLM into one machine.
# It takes the documents, pulls out their text, glues it together,
# ويقوم بـ "حشوها"
# and "stuffs" it into the {context} box. 
# The question goes into {input}.
# Then it sends the full prompt to the LLM and returns the answer.
document_chain = create_stuff_documents_chain(llm, prompt)
# document_chain

from langchain_core.documents import Document
document_chain.invoke({
    "input":"what is love?",
    "context":[Document(page_content="LOVE IS AWESOME LOLO, "
    "and makes the world a better place, Mohammad Is A good person, " \
    "Love is the best thing in the world, Love is the" \
    " most powerful force in the universe, Love is the answer to " \
    "all questions, Love is the key to happiness," \
    " Love is the meaning of life, Love is the reason we are " \
    "here, Love is the purpose of existence, Love is the ultimate " \
    "goal of humanity, Love is the essence of being, " \
    "Love is the foundation of all relationships, " \
)]
})

'Love is awesome and makes the world a better place; it is the best thing in the world, the most powerful force in the universe, and the answer to all questions. It is the key to happiness, the meaning of life, the reason we are here, the purpose of existence, the ultimate goal of humanity, the essence of being, and the foundation of all relationships.'

In [ ]:
# However, we want the documents 
# to first come from the retriever we just set up. 
# That way, we can use the retriever to dynamically 
# select the most relevant documents and pass those in for a given question.
from langchain_classic.chains import create_retrieval_chain

# RETRIEVER: a searcher built from our vector database.
# It takes the question, finds the 3 text pieces most similar to it (k=3),
# and returns them. It does NOT write the answer.
# Those pieces are then passed to the LLM as {context},
# and the LLM uses them to write the final answer.
retriever = vectorstoredb.as_retriever(search_kwargs={"k": 3})
retrieval_chain = create_retrieval_chain(retriever, document_chain)

response = retrieval_chain.invoke({"input": "What is organization?"})
# print(response["context"])
print(response["answer"])

An **organization** in LangSmith is a logical grouping of users that represents a company or a shared team. It sets up organization‑wide settings—such as user management, single sign‑on (SSO), OAuth provider configuration, custom role creation, billing, and usage tracking—that apply to every workspace inside that organization. Typically, there is one organization per company, but an organization can contain multiple workspaces. When you first log in, a personal organization is created automatically; you can then create a separate shared organization and invite teammates to collaborate.
